# 🚀 Session 4: Sending Data to APIs - Hands-On

Welcome to the interactive hands-on tutorial for Session 4! 

In this notebook, you will:
- 📝 Learn about GET vs POST requests
- 📦 Understand request bodies and Pydantic models
- ✅ Master data validation with constraints
- ⚠️ Implement proper error handling
- 🏗️ Build complete API endpoints
- 🧪 Test your endpoints with real data
- 💪 Complete hands-on exercises

**Goal:** By the end, you'll be able to build robust APIs that accept and validate user data safely.

Let's get started! 🎉

## 📚 Section 1: Import Required Libraries

Let's start by importing all the libraries we need for this session.

In [1]:
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field
from typing import Optional, List
from datetime import datetime
from fastapi.testclient import TestClient
import json

print("✅ All imports successful!")
print("Ready to build APIs that accept data.")


✅ All imports successful!
Ready to build APIs that accept data.


## 📦 Section 2: Pydantic Models - Request Bodies

Pydantic models define the structure of data sent to your API. Let's create our first model:

In [4]:
# Create Pydantic models for our API

class Product(BaseModel):
    """Basic product model"""
    name: str
    price: float
    description: Optional[str] = None
    in_stock: bool = True

class User(BaseModel):
    """User model with validation"""
    name: str = Field(..., min_length=1, max_length=100)
    email: str
    age: int = Field(..., ge=0, le=150)
    is_active: bool = True

class Order(BaseModel):
    """Order model with nested data"""
    product_name: str
    quantity: int = Field(..., ge=1)
    customer_email: str
    special_instructions: Optional[str] = None

# Test models
print("✅ Models created successfully!\n")

# Example of valid data
valid_product = Product(name="Laptop", price=999.99, description="High-performance laptop")
print(f"Valid Product: {valid_product}\n")

# Example with optional field
user_data = User(name="Alice", email="alice@example.com", age=28)
print(f"Valid User: {user_data}\n")

# Show the JSON representation
print(f"Product as JSON: {valid_product.model_dump_json()}")


✅ Models created successfully!

Valid Product: name='Laptop' price=999.99 description='High-performance laptop' in_stock=True

Valid User: name='Alice' email='alice@example.com' age=28 is_active=True

Product as JSON: {"name":"Laptop","price":999.99,"description":"High-performance laptop","in_stock":true}


## ✅ Section 3: Type Validation and Field Constraints

Pydantic automatically validates data types. Let's see how validation errors work:

In [5]:
# Test type validation
from pydantic import ValidationError

print("Testing Pydantic Validation:\n")
print("=" * 50)

# Test 1: Invalid type
print("\n1️⃣ Testing invalid type (string instead of int for age):")
try:
    invalid_user = User(name="Bob", email="bob@example.com", age="thirty")
except ValidationError as e:
    print(f"❌ Validation Error: {e.error_count()} error(s)")
    for error in e.errors():
        print(f"   Field: {error['loc'][0]}, Error: {error['msg']}")

# Test 2: Field constraint violation (age out of range)
print("\n2️⃣ Testing field constraint (age > 150):")
try:
    invalid_user = User(name="Charlie", email="charlie@example.com", age=200)
except ValidationError as e:
    print(f"❌ Validation Error: {e.error_count()} error(s)")
    for error in e.errors():
        print(f"   Field: {error['loc'][0]}, Error: {error['msg']}")

# Test 3: Name too long
print("\n3️⃣ Testing max_length constraint on name:")
try:
    long_name = "A" * 150
    invalid_user = User(name=long_name, email="dave@example.com", age=30)
except ValidationError as e:
    print(f"❌ Validation Error: {e.error_count()} error(s)")
    for error in e.errors():
        print(f"   Field: {error['loc'][0]}, Error: {error['msg']}")

# Test 4: Valid data
print("\n4️⃣ Testing valid data:")
valid_user = User(name="Eve", email="eve@example.com", age=35)
print(f"✅ User created: {valid_user}")

print("\n" + "=" * 50)
print("✅ Validation working correctly!")


Testing Pydantic Validation:


1️⃣ Testing invalid type (string instead of int for age):
❌ Validation Error: 1 error(s)
   Field: age, Error: Input should be a valid integer, unable to parse string as an integer

2️⃣ Testing field constraint (age > 150):
❌ Validation Error: 1 error(s)
   Field: age, Error: Input should be less than or equal to 150

3️⃣ Testing max_length constraint on name:
❌ Validation Error: 1 error(s)
   Field: name, Error: String should have at most 100 characters

4️⃣ Testing valid data:
✅ User created: name='Eve' email='eve@example.com' age=35 is_active=True

✅ Validation working correctly!


## 🏗️ Section 4: Building a Complete API with POST Endpoints

Let's create an API that accepts data through POST requests and handles errors properly:

In [6]:
# Create FastAPI app with POST endpoints
app = FastAPI(
    title="Data API",
    description="Learn to send and validate data",
    version="1.0.0"
)

# Mock database
users_db = {}
products_db = {}
next_user_id = 1
next_product_id = 1

# ========== USER ENDPOINTS ==========

@app.post("/users", status_code=201)
def create_user(user: User):
    """Create a new user"""
    global next_user_id
    
    # Business logic validation: check for duplicate email
    for existing_user in users_db.values():
        if existing_user['email'] == user.email:
            raise HTTPException(
                status_code=409,
                detail=f"Email {user.email} already registered"
            )
    
    # Create user
    user_id = next_user_id
    next_user_id += 1
    
    user_data = {
        'id': user_id,
        'name': user.name,
        'email': user.email,
        'age': user.age,
        'is_active': user.is_active,
        'created_at': datetime.now().isoformat()
    }
    
    users_db[user_id] = user_data
    return {"status": "created", "user": user_data, "status_code": 201}

@app.get("/users")
def list_users():
    """Get all users"""
    return {"total": len(users_db), "users": list(users_db.values())}

@app.get("/users/{user_id}")
def get_user(user_id: int):
    """Get a specific user"""
    if user_id not in users_db:
        raise HTTPException(
            status_code=404,
            detail=f"User with id {user_id} not found"
        )
    return users_db[user_id]

@app.put("/users/{user_id}")
def update_user(user_id: int, user: User):
    """Update a user"""
    if user_id not in users_db:
        raise HTTPException(
            status_code=404,
            detail=f"User with id {user_id} not found"
        )
    
    # Update fields
    users_db[user_id]['name'] = user.name
    users_db[user_id]['email'] = user.email
    users_db[user_id]['age'] = user.age
    users_db[user_id]['is_active'] = user.is_active
    
    return {"status": "updated", "user": users_db[user_id]}

@app.delete("/users/{user_id}", status_code=204)
def delete_user(user_id: int):
    """Delete a user"""
    if user_id not in users_db:
        raise HTTPException(
            status_code=404,
            detail=f"User with id {user_id} not found"
        )
    
    del users_db[user_id]
    return None

# ========== PRODUCT ENDPOINTS ==========

@app.post("/products", status_code=201)
def create_product(product: Product):
    """Create a new product"""
    global next_product_id
    
    product_id = next_product_id
    next_product_id += 1
    
    product_data = {
        'id': product_id,
        'name': product.name,
        'price': product.price,
        'description': product.description,
        'in_stock': product.in_stock,
        'created_at': datetime.now().isoformat()
    }
    
    products_db[product_id] = product_data
    return {"status": "created", "product": product_data, "status_code": 201}

@app.get("/products")
def list_products():
    """Get all products"""
    return {"total": len(products_db), "products": list(products_db.values())}

print("✅ API created with 7 endpoints!")
print("   - POST /users (create)")
print("   - GET /users (list)")
print("   - GET /users/{user_id} (get)")
print("   - PUT /users/{user_id} (update)")
print("   - DELETE /users/{user_id} (delete)")
print("   - POST /products (create)")
print("   - GET /products (list)")


✅ API created with 7 endpoints!
   - POST /users (create)
   - GET /users (list)
   - GET /users/{user_id} (get)
   - PUT /users/{user_id} (update)
   - DELETE /users/{user_id} (delete)
   - POST /products (create)
   - GET /products (list)


## 🧪 Section 5: Testing POST Requests with TestClient

Let's test our API endpoints to see how data flows through the system:

In [7]:
# Create test client
client = TestClient(app)

print("\n" + "="*60)
print("TESTING POST REQUESTS")
print("="*60)

# Test 1: Create a user (successful)
print("\n1️⃣ POST /users - Create a user (valid data)")
user_data = {
    "name": "Alice Johnson",
    "email": "alice@example.com",
    "age": 28,
    "is_active": True
}
response = client.post("/users", json=user_data)
print(f"Status: {response.status_code}")
print(f"Response: {response.json()}")
alice_id = response.json()['user']['id']


TESTING POST REQUESTS

1️⃣ POST /users - Create a user (valid data)
Status: 201
Response: {'status': 'created', 'user': {'id': 1, 'name': 'Alice Johnson', 'email': 'alice@example.com', 'age': 28, 'is_active': True, 'created_at': '2026-06-25T21:45:26.919208'}, 'status_code': 201}


In [8]:
# Test 2: Create another user
print("\n2️⃣ POST /users - Create another user")
user_data2 = {
    "name": "Bob Smith",
    "email": "bob@example.com",
    "age": 35,
    "is_active": True
}
response = client.post("/users", json=user_data2)
print(f"Status: {response.status_code}")
print(f"Response: {response.json()}")
bob_id = response.json()['user']['id']


2️⃣ POST /users - Create another user
Status: 201
Response: {'status': 'created', 'user': {'id': 2, 'name': 'Bob Smith', 'email': 'bob@example.com', 'age': 35, 'is_active': True, 'created_at': '2026-06-25T21:45:59.348715'}, 'status_code': 201}


In [9]:
# Test 3: Try to create duplicate email (error case)
print("\n3️⃣ POST /users - Create user with duplicate email (ERROR)")
duplicate_email = {
    "name": "Alice Copy",
    "email": "alice@example.com",
    "age": 30,
    "is_active": True
}
response = client.post("/users", json=duplicate_email)
print(f"Status: {response.status_code}")
print(f"Response: {response.json()}")


3️⃣ POST /users - Create user with duplicate email (ERROR)
Status: 409
Response: {'detail': 'Email alice@example.com already registered'}


In [10]:
# Test 4: Create user with invalid data (validation error)
print("\n4️⃣ POST /users - Create user with invalid age (ERROR)")
invalid_data = {
    "name": "Charlie",
    "email": "charlie@example.com",
    "age": 200,  # Out of range
    "is_active": True
}
response = client.post("/users", json=invalid_data)
print(f"Status: {response.status_code}")
error_detail = response.json()['detail'][0]
print(f"Error: {error_detail['msg']}")


4️⃣ POST /users - Create user with invalid age (ERROR)
Status: 422
Error: Input should be less than or equal to 150


In [11]:
# Test 5: Create user with wrong type
print("\n5️⃣ POST /users - Create user with string age (ERROR)")
wrong_type = {
    "name": "David",
    "email": "david@example.com",
    "age": "thirty",  # Should be int
    "is_active": True
}
response = client.post("/users", json=wrong_type)
print(f"Status: {response.status_code}")
error_detail = response.json()['detail'][0]
print(f"Error: {error_detail['msg']}")


5️⃣ POST /users - Create user with string age (ERROR)
Status: 422
Error: Input should be a valid integer, unable to parse string as an integer


In [12]:
# Test 6: Create a product
print("\n6️⃣ POST /products - Create a product")
product_data = {
    "name": "Laptop",
    "price": 999.99,
    "description": "High-performance laptop",
    "in_stock": True
}
response = client.post("/products", json=product_data)
print(f"Status: {response.status_code}")
print(f"Response: {response.json()}")


6️⃣ POST /products - Create a product
Status: 201
Response: {'status': 'created', 'product': {'id': 1, 'name': 'Laptop', 'price': 999.99, 'description': 'High-performance laptop', 'in_stock': True, 'created_at': '2026-06-25T21:49:30.839018'}, 'status_code': 201}


In [19]:
# Test 7: GET all users
print("\n7️⃣ GET /users - List all users")
response = client.get("/users")
print(f"Status: {response.status_code}")
data = response.json()
print(f"Total users: {data['total']}")
for user in data['users']:
    print(f"  - {user['name']} ({user['email']})")


7️⃣ GET /users - List all users
Status: 200
Total users: 1
  - Alice Updated (alice.updated@example.com)


In [14]:
# Test 8: GET specific user
print("\n8️⃣ GET /users/{user_id} - Get specific user")
response = client.get(f"/users/{alice_id}")
print(f"Status: {response.status_code}")
print(f"User: {response.json()}")


8️⃣ GET /users/{user_id} - Get specific user
Status: 200
User: {'id': 1, 'name': 'Alice Johnson', 'email': 'alice@example.com', 'age': 28, 'is_active': True, 'created_at': '2026-06-25T21:45:26.919208'}


In [16]:
# Test 9: UPDATE user
print("\n9️⃣ PUT /users/{user_id} - Update user")
update_data = {
    "name": "Alice Updated",
    "email": "alice.updated@example.com",
    "age": 30,
    "is_active": False
}
response = client.put(f"/users/{alice_id}", json=update_data)
print(f"Status: {response.status_code}")
print(f"Response: {response.json()}")


9️⃣ PUT /users/{user_id} - Update user
Status: 200
Response: {'status': 'updated', 'user': {'id': 1, 'name': 'Alice Updated', 'email': 'alice.updated@example.com', 'age': 30, 'is_active': False, 'created_at': '2026-06-25T21:45:26.919208'}}


In [17]:
# Test 10: DELETE user
print("\n🔟 DELETE /users/{user_id} - Delete user")
response = client.delete(f"/users/{bob_id}")
print(f"Status: {response.status_code}")
print(f"User {bob_id} deleted successfully")


🔟 DELETE /users/{user_id} - Delete user
Status: 204
User 2 deleted successfully


In [18]:
# Verify deletion
response = client.get(f"/users/{bob_id}")
print(f"Trying to get deleted user - Status: {response.status_code}")
print(f"Error: {response.json()['detail']}")

print("\n" + "="*60)
print("✅ All tests completed!")
print("="*60)


Trying to get deleted user - Status: 404
Error: User with id 2 not found

✅ All tests completed!


In [ ]:
# Create test client
client = TestClient(app)

print("\n" + "="*60)
print("TESTING POST REQUESTS")
print("="*60)

# Test 1: Create a user (successful)
print("\n1️⃣ POST /users - Create a user (valid data)")
user_data = {
    "name": "Alice Johnson",
    "email": "alice@example.com",
    "age": 28,
    "is_active": True
}
response = client.post("/users", json=user_data)
print(f"Status: {response.status_code}")
print(f"Response: {response.json()}")
alice_id = response.json()['user']['id']

# Test 2: Create another user
print("\n2️⃣ POST /users - Create another user")
user_data2 = {
    "name": "Bob Smith",
    "email": "bob@example.com",
    "age": 35,
    "is_active": True
}
response = client.post("/users", json=user_data2)
print(f"Status: {response.status_code}")
print(f"Response: {response.json()}")
bob_id = response.json()['user']['id']

# Test 3: Try to create duplicate email (error case)
print("\n3️⃣ POST /users - Create user with duplicate email (ERROR)")
duplicate_email = {
    "name": "Alice Copy",
    "email": "alice@example.com",
    "age": 30,
    "is_active": True
}
response = client.post("/users", json=duplicate_email)
print(f"Status: {response.status_code}")
print(f"Response: {response.json()}")

# Test 4: Create user with invalid data (validation error)
print("\n4️⃣ POST /users - Create user with invalid age (ERROR)")
invalid_data = {
    "name": "Charlie",
    "email": "charlie@example.com",
    "age": 200,  # Out of range
    "is_active": True
}
response = client.post("/users", json=invalid_data)
print(f"Status: {response.status_code}")
error_detail = response.json()['detail'][0]
print(f"Error: {error_detail['msg']}")

# Test 5: Create user with wrong type
print("\n5️⃣ POST /users - Create user with string age (ERROR)")
wrong_type = {
    "name": "David",
    "email": "david@example.com",
    "age": "thirty",  # Should be int
    "is_active": True
}
response = client.post("/users", json=wrong_type)
print(f"Status: {response.status_code}")
error_detail = response.json()['detail'][0]
print(f"Error: {error_detail['msg']}")

# Test 6: Create a product
print("\n6️⃣ POST /products - Create a product")
product_data = {
    "name": "Laptop",
    "price": 999.99,
    "description": "High-performance laptop",
    "in_stock": True
}
response = client.post("/products", json=product_data)
print(f"Status: {response.status_code}")
print(f"Response: {response.json()}")

# Test 7: GET all users
print("\n7️⃣ GET /users - List all users")
response = client.get("/users")
print(f"Status: {response.status_code}")
data = response.json()
print(f"Total users: {data['total']}")
for user in data['users']:
    print(f"  - {user['name']} ({user['email']})")

# Test 8: GET specific user
print("\n8️⃣ GET /users/{user_id} - Get specific user")
response = client.get(f"/users/{alice_id}")
print(f"Status: {response.status_code}")
print(f"User: {response.json()}")

# Test 9: UPDATE user
print("\n9️⃣ PUT /users/{user_id} - Update user")
update_data = {
    "name": "Alice Updated",
    "email": "alice.updated@example.com",
    "age": 29,
    "is_active": True
}
response = client.put(f"/users/{alice_id}", json=update_data)
print(f"Status: {response.status_code}")
print(f"Response: {response.json()}")

# Test 10: DELETE user
print("\n🔟 DELETE /users/{user_id} - Delete user")
response = client.delete(f"/users/{bob_id}")
print(f"Status: {response.status_code}")
print(f"User {bob_id} deleted successfully")

# Verify deletion
response = client.get(f"/users/{bob_id}")
print(f"Trying to get deleted user - Status: {response.status_code}")
print(f"Error: {response.json()['detail']}")

print("\n" + "="*60)
print("✅ All tests completed!")
print("="*60)


## 🎯 Section 6: Advanced Patterns - PATCH and Bulk Operations

Let's implement more sophisticated API patterns:

In [ ]:
# Define models for advanced patterns

class UserUpdate(BaseModel):
    """Partial update model - all fields optional"""
    name: Optional[str] = None
    email: Optional[str] = None
    age: Optional[int] = None
    is_active: Optional[bool] = None

class BulkUserCreate(BaseModel):
    """Model for creating multiple users at once"""
    users: List[User]

# Add PATCH endpoint for partial updates
@app.patch("/users/{user_id}")
def partial_update_user(user_id: int, user_update: UserUpdate):
    """Partially update user - only provided fields are updated"""
    if user_id not in users_db:
        raise HTTPException(
            status_code=404,
            detail=f"User with id {user_id} not found"
        )
    
    # Only update fields that were provided
    if user_update.name is not None:
        users_db[user_id]['name'] = user_update.name
    if user_update.email is not None:
        users_db[user_id]['email'] = user_update.email
    if user_update.age is not None:
        users_db[user_id]['age'] = user_update.age
    if user_update.is_active is not None:
        users_db[user_id]['is_active'] = user_update.is_active
    
    return {"status": "updated", "user": users_db[user_id]}

# Add bulk create endpoint
@app.post("/users/bulk/create", status_code=201)
def create_users_bulk(bulk_data: BulkUserCreate):
    """Create multiple users in one request"""
    global next_user_id
    
    created_users = []
    errors = []
    
    for user in bulk_data.users:
        try:
            # Check for duplicates
            for existing_user in users_db.values():
                if existing_user['email'] == user.email:
                    errors.append({
                        "email": user.email,
                        "error": "Email already exists"
                    })
                    continue
            
            # Create user
            user_id = next_user_id
            next_user_id += 1
            
            user_data = {
                'id': user_id,
                'name': user.name,
                'email': user.email,
                'age': user.age,
                'is_active': user.is_active,
                'created_at': datetime.now().isoformat()
            }
            
            users_db[user_id] = user_data
            created_users.append(user_data)
        except Exception as e:
            errors.append({
                "email": user.email,
                "error": str(e)
            })
    
    return {
        "created_count": len(created_users),
        "error_count": len(errors),
        "created_users": created_users,
        "errors": errors
    }

print("✅ Advanced patterns added!")
print("   - PATCH /users/{user_id} (partial update)")
print("   - POST /users/bulk/create (bulk create)")

# Test PATCH
print("\n" + "="*60)
print("TESTING ADVANCED PATTERNS")
print("="*60)

# Create a test user
print("\n1️⃣ Create a test user first")
new_user = {
    "name": "Test User",
    "email": "test@example.com",
    "age": 25,
    "is_active": True
}
response = client.post("/users", json=new_user)
test_user_id = response.json()['user']['id']
print(f"Created user {test_user_id}")

# Test PATCH - update only name
print("\n2️⃣ PATCH /users/{user_id} - Update only name")
patch_data = {"name": "Updated Name Only"}
response = client.patch(f"/users/{test_user_id}", json=patch_data)
print(f"Status: {response.status_code}")
print(f"Updated: {response.json()['user']}")

# Test PATCH - update only is_active
print("\n3️⃣ PATCH /users/{user_id} - Update only is_active")
patch_data = {"is_active": False}
response = client.patch(f"/users/{test_user_id}", json=patch_data)
print(f"Status: {response.status_code}")
print(f"Updated: {response.json()['user']}")

# Test bulk create
print("\n4️⃣ POST /users/bulk/create - Create multiple users")
bulk_data = {
    "users": [
        {
            "name": "Bulk User 1",
            "email": "bulk1@example.com",
            "age": 30,
            "is_active": True
        },
        {
            "name": "Bulk User 2",
            "email": "bulk2@example.com",
            "age": 32,
            "is_active": True
        },
        {
            "name": "Bulk User 3",
            "email": "bulk3@example.com",
            "age": 28,
            "is_active": True
        }
    ]
}
response = client.post("/users/bulk/create", json=bulk_data)
print(f"Status: {response.status_code}")
data = response.json()
print(f"Created: {data['created_count']} users")
print(f"Errors: {data['error_count']}")
for user in data['created_users']:
    print(f"  - {user['name']} ({user['email']})")

# Test bulk create with duplicate
print("\n5️⃣ POST /users/bulk/create - Bulk with duplicate email")
bulk_with_dup = {
    "users": [
        {
            "name": "New User",
            "email": "new@example.com",
            "age": 40,
            "is_active": True
        },
        {
            "name": "Duplicate",
            "email": "bulk1@example.com",  # Already exists
            "age": 35,
            "is_active": True
        }
    ]
}
response = client.post("/users/bulk/create", json=bulk_with_dup)
print(f"Status: {response.status_code}")
data = response.json()
print(f"Created: {data['created_count']} users")
print(f"Errors: {data['error_count']}")

print("\n" + "="*60)
print("✅ Advanced patterns tested!")
print("="*60)


## 💪 Section 7: Hands-On Exercises

Now it's your turn! Try to implement these features. Start with Exercise 1 and work your way up.

### Exercise 1: Create a Comment Model and Endpoint ⭐
Create a `Comment` model with:
- `text`: str (required, min 1 char, max 500 chars)
- `author_email`: str (required)
- `rating`: int (required, between 1-5)

Add a POST endpoint `/comments` that:
- Creates a new comment
- Returns status 201
- Validates all fields

**Hint:** Use `Field(...)` with `min_length`, `max_length`, `ge`, `le` constraints

### Exercise 2: Add Error Handling ⭐⭐
Modify the comment endpoint to:
- Return 400 if rating is not between 1-5
- Return 400 if text is empty
- Return 400 if email is invalid format

### Exercise 3: Create a Review Bulk Endpoint ⭐⭐⭐
Create a model for bulk comment creation and add endpoint:
- POST `/comments/bulk` that accepts multiple comments
- Returns count of created vs errors
- Handle duplicates/errors gracefully

### Exercise 4: Add PATCH for Comments ⭐⭐⭐
Create a `CommentUpdate` model (all optional fields) and add:
- PATCH `/comments/{comment_id}`
- Only update provided fields
- Validate new data

### Exercise 5: Advanced - Search by Rating ⭐⭐⭐⭐
Add endpoint:
- GET `/comments/search?min_rating=3&max_rating=5`
- Filter comments by rating range
- Return matching comments
- Handle invalid ranges with error response

In [ ]:
# Exercise 1: Create Comment Model and Endpoint
print("=" * 60)
print("EXERCISE 1: Comment Model and Endpoint")
print("=" * 60)

# TODO: Define Comment model here
# class Comment(BaseModel):
#     text: str = Field(...)  # Add constraints
#     author_email: str
#     rating: int = Field(...)  # Add constraints (1-5)

# Mock comments database
comments_db = {}
next_comment_id = 1

# TODO: Create POST /comments endpoint
# @app.post("/comments", status_code=201)
# def create_comment(comment: Comment):
#     # Your code here
#     pass

# For now, here's a complete solution to learn from:
print("\n✅ SOLUTION FOR EXERCISE 1:\n")

class Comment(BaseModel):
    text: str = Field(..., min_length=1, max_length=500)
    author_email: str
    rating: int = Field(..., ge=1, le=5)

@app.post("/comments", status_code=201)
def create_comment(comment: Comment):
    """Create a new comment"""
    global next_comment_id
    
    comment_id = next_comment_id
    next_comment_id += 1
    
    comment_data = {
        'id': comment_id,
        'text': comment.text,
        'author_email': comment.author_email,
        'rating': comment.rating,
        'created_at': datetime.now().isoformat()
    }
    
    comments_db[comment_id] = comment_data
    return {"status": "created", "comment": comment_data, "status_code": 201}

# Test Exercise 1
print("Testing Exercise 1:")
comment = {
    "text": "Great product!",
    "author_email": "user@example.com",
    "rating": 5
}
response = client.post("/comments", json=comment)
print(f"Status: {response.status_code}")
print(f"Response: {response.json()}")

# Test validation
print("\nTesting validation (invalid rating):")
invalid_comment = {
    "text": "Bad",
    "author_email": "user@example.com",
    "rating": 10  # Out of range
}
response = client.post("/comments", json=invalid_comment)
print(f"Status: {response.status_code}")
if response.status_code != 201:
    print(f"Error: {response.json()['detail'][0]['msg']}")

print("\n✅ Exercise 1 complete!")


In [ ]:
# Exercise 2 & 3 & 4 Solutions (demonstrating more advanced patterns)

print("\n" + "=" * 60)
print("EXERCISE 2-4: Solutions")
print("=" * 60)

# Exercise 3: Bulk Comments
class BulkCommentCreate(BaseModel):
    comments: List[Comment]

@app.post("/comments/bulk", status_code=201)
def create_comments_bulk(bulk_data: BulkCommentCreate):
    """Create multiple comments"""
    global next_comment_id
    
    created = []
    errors = []
    
    for comment in bulk_data.comments:
        try:
            comment_id = next_comment_id
            next_comment_id += 1
            
            comment_data = {
                'id': comment_id,
                'text': comment.text,
                'author_email': comment.author_email,
                'rating': comment.rating,
                'created_at': datetime.now().isoformat()
            }
            
            comments_db[comment_id] = comment_data
            created.append(comment_data)
        except Exception as e:
            errors.append({
                "comment": comment.text[:50],
                "error": str(e)
            })
    
    return {
        "created_count": len(created),
        "error_count": len(errors),
        "comments": created
    }

# Exercise 4: PATCH for Comments
class CommentUpdate(BaseModel):
    text: Optional[str] = Field(None, min_length=1, max_length=500)
    rating: Optional[int] = Field(None, ge=1, le=5)

@app.patch("/comments/{comment_id}")
def update_comment(comment_id: int, comment_update: CommentUpdate):
    """Update a comment"""
    if comment_id not in comments_db:
        raise HTTPException(
            status_code=404,
            detail=f"Comment {comment_id} not found"
        )
    
    if comment_update.text is not None:
        comments_db[comment_id]['text'] = comment_update.text
    if comment_update.rating is not None:
        comments_db[comment_id]['rating'] = comment_update.rating
    
    return {"status": "updated", "comment": comments_db[comment_id]}

# Exercise 5: Search by Rating
@app.get("/comments/search")
def search_comments(min_rating: int = 1, max_rating: int = 5):
    """Search comments by rating range"""
    if min_rating < 1 or max_rating > 5 or min_rating > max_rating:
        raise HTTPException(
            status_code=400,
            detail="min_rating must be 1-5, max_rating must be 1-5, and min <= max"
        )
    
    matching = [
        c for c in comments_db.values()
        if min_rating <= c['rating'] <= max_rating
    ]
    
    return {
        "query": {"min_rating": min_rating, "max_rating": max_rating},
        "count": len(matching),
        "comments": matching
    }

print("✅ All exercises implemented!")

# Test them
print("\nTesting Exercise 3 (Bulk Create):")
bulk = {
    "comments": [
        {"text": "Excellent!", "author_email": "a@x.com", "rating": 5},
        {"text": "Good", "author_email": "b@x.com", "rating": 4},
        {"text": "OK", "author_email": "c@x.com", "rating": 3}
    ]
}
response = client.post("/comments/bulk", json=bulk)
data = response.json()
print(f"Created: {data['created_count']} comments")

print("\nTesting Exercise 4 (PATCH):")
patch_update = {"rating": 4}
response = client.patch("/comments/1", json=patch_update)
print(f"Status: {response.status_code}")
print(f"Updated: {response.json()['comment']}")

print("\nTesting Exercise 5 (Search by Rating):")
response = client.get("/comments/search?min_rating=4&max_rating=5")
data = response.json()
print(f"Found {data['count']} comments with rating 4-5")

print("\n✅ All exercises tested successfully!")


## 📚 Summary: Key Concepts from Session 4

### ✅ What You Learned:

1. **GET vs POST**
   - GET: Retrieve data from URL parameters
   - POST: Send data in request body to create resources

2. **Request Bodies**
   - Defined using Pydantic models
   - Automatically validated and converted from JSON

3. **Pydantic Validation**
   - Type checking (str, int, float, bool, etc.)
   - Field constraints (min/max, ge/le, regex)
   - Optional fields with defaults
   - Nested models

4. **Error Handling**
   - 201: Created (POST success)
   - 400: Bad Request (validation error)
   - 404: Not Found
   - 409: Conflict (duplicate)
   - 422: Unprocessable Entity (Pydantic validation)

5. **CRUD Operations**
   - Create: POST with status 201
   - Read: GET specific or list
   - Update: PUT (replace) or PATCH (partial)
   - Delete: DELETE with status 204

6. **Advanced Patterns**
   - Partial updates with PATCH
   - Bulk operations
   - Response filtering
   - Search and filtering

### 🎯 Best Practices:

- ✅ Always validate input data
- ✅ Return appropriate HTTP status codes
- ✅ Handle business logic errors (duplicates, constraints)
- ✅ Never expose sensitive data in responses
- ✅ Test your endpoints with various scenarios
- ✅ Use type hints for automatic validation
- ✅ Document your endpoints

### 🚀 Next Steps:

1. Save your code to a `main.py` file
2. Run: `uvicorn main:app --reload`
3. Visit: `http://localhost:8000/docs`
4. Try endpoints in Swagger UI
5. Build more endpoints with validation
6. Connect to a real database (PostgreSQL, MongoDB)
7. Add authentication and authorization

### 📖 Resources:

- **FastAPI Docs:** https://fastapi.tiangolo.com/
- **Pydantic Docs:** https://docs.pydantic.dev/
- **HTTP Status Codes:** https://httpwg.org/specs/rfc7231.html#status.codes
- **REST Best Practices:** https://restfulapi.net/

Happy coding! 🎉

In [ ]:
# Quick Reference: Session 4 Patterns

reference = """
╔════════════════════════════════════════════════════════════════════════════╗
║                   SESSION 4: QUICK REFERENCE GUIDE                         ║
╚════════════════════════════════════════════════════════════════════════════╝

1. BASIC PYDANTIC MODEL
   ─────────────────────────────────────────────────────────────────────────
   class Item(BaseModel):
       name: str
       price: float
       description: Optional[str] = None

2. MODEL WITH VALIDATION
   ─────────────────────────────────────────────────────────────────────────
   from pydantic import Field
   
   class Product(BaseModel):
       name: str = Field(..., min_length=1, max_length=100)
       price: float = Field(..., gt=0)
       stock: int = Field(default=0, ge=0)

3. POST ENDPOINT - CREATE
   ─────────────────────────────────────────────────────────────────────────
   @app.post("/items", status_code=201)
   def create_item(item: Item):
       # item is automatically validated
       return {"created": True, "item": item}

4. POST WITH ERROR HANDLING
   ─────────────────────────────────────────────────────────────────────────
   @app.post("/users")
   def create_user(user: User):
       if user_exists(user.email):
           raise HTTPException(status_code=409, detail="Email taken")
       return create_in_db(user)

5. PATCH - PARTIAL UPDATE
   ─────────────────────────────────────────────────────────────────────────
   class ItemUpdate(BaseModel):
       name: Optional[str] = None
       price: Optional[float] = None
   
   @app.patch("/items/{item_id}")
   def update_item(item_id: int, update: ItemUpdate):
       # Only update provided fields
       if update.name:
           item.name = update.name
       if update.price:
           item.price = update.price
       return item

6. BULK CREATE
   ─────────────────────────────────────────────────────────────────────────
   class BulkCreate(BaseModel):
       items: List[Item]
   
   @app.post("/items/bulk")
   def bulk_create(bulk: BulkCreate):
       created = []
       for item in bulk.items:
           created.append(save(item))
       return {"created": len(created)}

7. HTTP STATUS CODES
   ─────────────────────────────────────────────────────────────────────────
   200: OK (normal success)
   201: Created (POST/bulk success)
   204: No Content (DELETE)
   400: Bad Request (validation/logic error)
   409: Conflict (duplicate, already exists)
   422: Unprocessable Entity (Pydantic error)
   
8. VALIDATION CONSTRAINTS
   ─────────────────────────────────────────────────────────────────────────
   min_length: Minimum string length
   max_length: Maximum string length
   gt: Greater than (>)
   ge: Greater or equal (>=)
   lt: Less than (<)
   le: Less or equal (<=)
   regex: Pattern matching
   
9. OPTIONAL VS REQUIRED
   ─────────────────────────────────────────────────────────────────────────
   name: str                    # Required
   name: Optional[str] = None   # Optional, no default
   name: str = "default"        # Optional with default
   name: Optional[str] = Field(None, min_length=1)  # Optional + validation

10. NESTED MODELS
    ─────────────────────────────────────────────────────────────────────────
    class Address(BaseModel):
        street: str
        city: str
    
    class User(BaseModel):
        name: str
        address: Address  # Nested
    
    # JSON: {"name": "John", "address": {"street": "123 Main", "city": "NYC"}}

╔════════════════════════════════════════════════════════════════════════════╗
║                          COMMON PATTERNS                                    ║
╚════════════════════════════════════════════════════════════════════════════╝

Pattern 1: Check Duplicate
    existing = db.filter(email=user.email)
    if existing:
        raise HTTPException(409, "Email taken")

Pattern 2: Validate Business Logic
    if user.age < 18:
        raise HTTPException(400, "Must be 18+")

Pattern 3: Handle Errors in Bulk
    errors = []
    for item in items:
        try:
            create(item)
        except Exception as e:
            errors.append(str(e))
    return {"created": len(success), "errors": len(errors)}

Pattern 4: Return Different Fields
    @app.get("/users", response_model=List[UserResponse])
    # Only returns specified fields, not password

Pattern 5: Search/Filter
    @app.get("/items/search")
    def search(min_price: float = 0, max_price: float = 1000):
        return [i for i in items if min_price <= i.price <= max_price]

╔════════════════════════════════════════════════════════════════════════════╗
║                          TESTING EXAMPLES                                   ║
╚════════════════════════════════════════════════════════════════════════════╝

client = TestClient(app)

# POST
response = client.post("/items", json={"name": "Item", "price": 9.99})
assert response.status_code == 201

# PATCH
response = client.patch("/items/1", json={"name": "Updated"})
assert response.status_code == 200

# Error handling
response = client.post("/items", json={"name": "Item"})  # Missing price
assert response.status_code == 422

# Verify response
data = response.json()
assert data["status"] == "created"
"""

print(reference)
print("\n✅ Use this guide as a reference for building your APIs!")
